# Graxella — the developer guide

**One notebook, the full substrate.** Every section follows the same shape:

> **WITHOUT graxella** (what you'd write and pay by hand) → **WITH graxella** → **what you gained, measured**.

## What graxella is

A governance substrate for multi-agent systems. Your agents stay framework-native
(LangChain / LangGraph / CrewAI / bare callables); graxella wraps them with a
memory-backed ledger, deterministic routing, self-healing tool dispatch, and an
evidence gate — so the system *learns*, and every learned change is **cited,
reviewable, and replayable**.

## The differentiators you will measure below

| # | Differentiator | Where |
|---|---|---|
| 1 | **Observability for free** — every tool call becomes a cited ledger row | Part 2 |
| 2 | **Fail once, learn forever** — drifted APIs heal with exactly ONE LLM call, ever | Part 3 |
| 3 | **Governed learning** — nothing changes runtime behavior without gate + citation | Part 4 |
| 4 | **Cited trust** — tool reliability scores you can audit down to the assertion | Part 5 |
| 5 | **Zero-token routing** — agent dispatch without a runtime LLM | Part 6 |
| 6 | **Bounded trajectories** — multi-hop with hard budgets and loop detection | Part 7 |
| 7 | **Value ledger** — tokens, cost, latency, ok-rate recomputed from assertions alone | Part 8 |
| 8 | **Operator UI** — Swagger + live topology map over the running system | Part 9 |

## Requirements

- `uv sync` at the repo root (installs graxella + siblings editable).
- **Optional**: [Ollama](https://ollama.com) serving `qwen2.5:3b` — enables Part 10
  (real-LLM finale). Everything else runs without any LLM.
  Set env `GRAXELLA_GUIDE_OFFLINE=1` to force the no-LLM path.

In [ ]:
import json, os, pprint, tempfile, time, urllib.request

import graxella

MODEL = "qwen2.5:3b"

def _ollama_up() -> bool:
    if os.environ.get("GRAXELLA_GUIDE_OFFLINE"):
        return False
    try:
        with urllib.request.urlopen("http://localhost:11434/api/tags", timeout=3) as r:
            models = [m["name"] for m in json.load(r)["models"]]
        return any(m.startswith(MODEL.split(":")[0]) for m in models)
    except Exception:
        return False

OLLAMA = _ollama_up()
print(f"graxella {graxella.__version__} — Ollama with {MODEL}: {OLLAMA}")

## Part 1 — `graxella.Session`: one object owns the substrate

Everything in graxella needs the same five things threaded through it:
**memory** (the ledger), **rulebook** (approved changes), **gate** (the reviewer),
**domain** (evidence scope), **model_id** (evidence is per-model). A `Session`
builds them once and every decorator/verb below reuses them.

| Parameter | Meaning |
|---|---|
| `name` | names the workdir and the ledger's `agent_id` |
| `domain` | evidence scope — trust and gate priors never leak across domains |
| `model_id` | evidence scope per model — swap models, keep honest priors |
| `workdir` | where the substrate lives: `mnema.db` (WAL-buffered SQLite ledger), `rulebook.json`, `routes.jsonl`. Default `.graxella/<name>/` **persists across runs — that's the point**. `"ephemeral"` for a loud throwaway. |

`grx` below is just a variable name — call it anything. The explicit primitives
stay reachable underneath: `grx.memory`, `grx.rulebook`, `grx.gate`.

In [ ]:
# a throwaway workdir so this guide is reproducible from zero every run;
# in production you'd omit workdir= and let evidence accumulate in .graxella/
grx = graxella.Session("dev-guide", domain="guide", model_id=MODEL,
                       workdir=tempfile.mkdtemp(prefix="graxella-guide-"))
print(grx)
print("workdir:", grx.workdir, " (files appear lazily on first write)")

## Part 2 — `@grx.tool`: observability for free

**What the decorator does.** It returns a *real* LangChain `StructuredTool`
(argument schema inferred from your signature — it drops into `create_agent`
unchanged). Inside sits a `ToolInterceptor`: every call records a **decision**
assertion (what was called, why) and an **outcome** assertion (ok/failed,
latency, error class) into the SQLite ledger. No logging code, no metrics
wiring, no APM agent.

In [ ]:
# WITHOUT graxella: a plain function. It runs — and leaves no trace.
def check_order_plain(order_id: str) -> str:
    return f"order {order_id}: espresso machine, delivered"

check_order_plain("1042")
print("history of calls?      none")
print("failure rate?          unknown")
print("latency distribution?  unknown — you'd wire logging + metrics per tool, by hand")

In [ ]:
# WITH graxella: same function, one decorator.
ORDERS = {"1042": {"item": "espresso machine", "days_ago": 6},
          "2077": {"item": "desk lamp", "days_ago": 40}}

@grx.tool
def check_order(order_id: str) -> str:
    """look up an order: item, delivery status and age in days"""
    o = ORDERS.get(order_id.strip())
    return (f"order {order_id}: {o['item']}, delivered, {o['days_ago']} days ago"
            if o else f"order {order_id}: not found")

print("what came back:", type(check_order).__name__, "| schema:", check_order.args)
print(check_order.invoke({"order_id": "1042"}))
print(check_order.invoke({"order_id": "7777"}))

In [ ]:
# THE GAIN, MEASURED: each call above became a decision + an outcome on the ledger.
rows = grx.memory.beliefs(predicate="outcome")
print(f"outcome assertions on the ledger: {len(rows)}")
pprint.pp(grx.stats()["total"])

## Part 3 — drift and the heal ladder: *fail once, learn forever*

**The scenario.** An internal API you depend on silently changed its schema last
sprint (`order_id` → `order_ref`). Nobody told your agents.

**The heal ladder** — cheapest rung first — runs inside every
`@grx.tool(fallback=...)`:

| Rung | What happens | LLM cost |
|---|---|---|
| 1. happy path | primary succeeds | 0 |
| 2. **promoted heal** | the rulebook holds a gate-approved transform → apply it, call the fallback | 0 |
| 2.5 proposed heal | a recipe was proposed this process and awaits review → reuse it deterministically | 0 |
| 3. **heal-once** | no rule yet → the **healer** proposes a `TransformRecipe` ONCE; it ships as a gated Proposal | 1 call, ever |
| 4. loud failure | nothing worked → typed failure outcome, re-raise | 0 |

**`@grx.healer`** registers the session-wide healer — **the one place an LLM may
appear in healing**. Whatever proposes the fix, the *output* is a deterministic,
diff-able artifact (`TransformRecipe`), never a prompt.

In [ ]:
# WITHOUT graxella: every drifted call needs the LLM to figure out the fix — again and again.
class CountingLLMFixer:
    calls = 0
    def fix(self, error: str, args: dict) -> dict:
        CountingLLMFixer.calls += 1        # imagine ~500 tokens per call, forever
        return {"order_ref": args["order_id"]}

def shipping_v1(args: dict) -> str:
    raise RuntimeError("HTTP_410_GONE: schema deprecated, use shipping.v2")

def shipping_v2(args: dict) -> str:
    return f"shipment for order {args['order_ref']}: delivered, signed"

fixer = CountingLLMFixer()
for i in range(5):
    try:
        shipping_v1({"order_id": str(1000 + i)})
    except RuntimeError as err:
        shipping_v2(fixer.fix(str(err), {"order_id": str(1000 + i)}))

print(f"5 calls -> LLM fixer invoked {CountingLLMFixer.calls} times (every call, forever)")

In [ ]:
# WITH graxella: register a healer once, declare the fallback, done.
@grx.healer
def field_rename_healer(tool_name: str, args: dict, error: str):
    # In production this is an LLM (08_refund_desk.py runs a real qwen2.5:3b here).
    # Deterministic in this guide so it's reproducible offline — the point stands
    # either way: whatever proposes it, the OUTPUT is a reviewable artifact.
    print(f"   [healer invoked for {tool_name}: {error[:45]}...]")
    return graxella.TransformRecipe(field_map={"order_id": "order_ref"})

@grx.tool(fallback=shipping_v2)
def get_shipping_status(order_id: str) -> str:
    """get the live shipping status for an order"""
    # the legacy v1 endpoint, silently retired last sprint:
    raise RuntimeError("HTTP_410_GONE: schema deprecated, use shipping.v2")

for oid in ("1042", "2077", "3001", "3002", "3003"):
    print(get_shipping_status.invoke({"order_id": oid}))

print(f"\n5 calls -> healer invoked {grx.healer_calls} time(s). Ever.  (naive: {CountingLLMFixer.calls})")

**What just happened, rung by rung:** call 1 hit rung 3 (heal-once): the healer
proposed `{order_id -> order_ref}`, the healed call succeeded, and the recipe
shipped as a **Proposal** with paired-replay evidence into the gate. Calls 2–5
hit rung 2.5: the proposed recipe was reused **deterministically** while the
proposal awaits review. With a real LLM healer that is 1×~500 tokens instead of
5×~500 — and the gap widens with every future call.

## Part 4 — the Evidence Gate: learning is governed, not silent

A system that silently rewrites its own dispatch is a compliance nightmare. So
in graxella **nothing changes runtime behavior without passing the gate**:

- The gate computes a **Bayesian posterior** from prior outcomes for the
  `(domain, kind, tool, model)` tuple. Warm tuples with strong evidence can
  auto-approve; **cold tuples always go to a human**.
- A human can approve — but **never override a constitution hard-block**
  (constitution over people, people over uncertainty). Rejections are always honored.
- Every verdict is itself a **cited ledger assertion**: the audit trail of the
  governance system is produced by the governance system.

The queue survives restarts — it's read from the ledger, not from process state.

In [ ]:
pend = grx.pending()          # the cross-process human-review queue
pprint.pp(pend)

In [ ]:
print(grx.why(pend[0]))       # the rendered verdict, before any human decision

In [ ]:
# approve: records the human decision -> re-decides through the gate -> promotes.
rule = grx.approve(pend[0], by="operator:you", note="field rename verified in review")
print("promoted:", type(rule).__name__)
print("\nrulebook.json now holds the artifact:")
print((grx.workdir / "rulebook.json").read_text()[:500], "...")

In [ ]:
# proof: the tool now heals via the PROMOTED rule (rung 2) — deterministic, cited.
print(get_shipping_status.invoke({"order_id": "9999"}))
print(f"healer calls, still: {grx.healer_calls}")
print("\nthe verdict after review:")
print(grx.why(pend[0]))

**Without graxella there is no artifact.** The "fix" lives inside an LLM retry
loop: unauditable, unreviewable, re-paid on every call, and gone when the
process dies. Here you have `rulebook.json` in version control, a gate verdict
with citations, and the operator's decision on the ledger — forever.

## Part 5 — cited tool trust

Trust is computed from the outcome ledger (Laplace-smoothed: `(s+1)/(s+f+2)`),
and every score carries **citations** — the assertion ids of the outcomes it was
computed from. A dashboard number you can audit down to the row.

In [ ]:
from graxella import tool_trust

for name, t in sorted(tool_trust(grx.memory, domain="guide").items()):
    print(f"{name:22s} score={t.score:.2f}  ok={t.successes:>2} fail={t.failures:>2}  "
          f"citations={len(t.citations)}")

## Part 6 — the mesh: routing without burning tokens

**The pattern everyone ships** (langgraph-supervisor): a supervisor **LLM** reads
the peer list and picks the next agent — one LLM call *per hop*, non-deterministic,
and "why did it route there?" has no answer beyond the prompt.

**`grx.mesh(agents)`**: deterministic scoring (TF-IDF by default;
`router="transformer"` upgrades to MiniLM embeddings — still zero tokens at
dispatch). Every agent gets a **peer directory** system message so it knows its
neighbours' capabilities and can end its reply with a typed
`HANDOFF: <peer> :: <task>` marker. Every route decision lands on the ledger.

*Anything is an agent*: bare callables, LangGraph graphs from `create_agent`,
CrewAI-shaped objects. (Want LLM routing anyway? `grx.supervisor(agents, model)` —
same governance, and it falls back loudly, never silently.)

In [ ]:
def billing(task: str) -> str:
    """handle billing refunds and payment questions"""
    return f"[billing] resolved: {task}"

def tech_support(task: str) -> str:
    """troubleshoot device setup and technical problems"""
    return f"[tech_support] resolved: {task}"

app = grx.mesh([billing, tech_support])

for task in ("refunds question about billing on order 1042",
             "troubleshoot my espresso machine setup problems"):
    route = app.invoke(task)["route"]
    print(f"{task[:46]:48s} -> {route['agent']:13s} score={route['score']:.2f}"
          f"  decision={route['decision_id'][:14]}...  routing tokens: 0")

Each route carries a `decision_id` — a ledger assertion you can replay and
explain. Two tasks routed, **zero tokens spent deciding**; the supervisor-LLM
pattern would have paid two LLM calls for a worse audit trail.

## Part 7 — trajectories: multi-hop, bounded by construction

`app.run_trajectory(task, ...)` runs the route → respond → HANDOFF loop with
**hard budgets** (`max_hops`, `max_tokens`, `max_wallclock_s`), loop detection
(same agent + same task signature twice = stop), and a typed result:
`completed | loop_detected | budget_exhausted | failed`. Runaway agent
ping-pong is impossible by construction, not by hope.

In [ ]:
t = app.run_trajectory("handle billing refunds for order 2077", max_hops=2)
print("status:", t.status, "| hops:", [h.agent for h in t.hops])
print("reply :", t.final_response[:100])

## Part 8 — the ledger answers: value accounting

`grx.stats()` recomputes everything from ledger assertions alone — count,
ok-rate, tokens in/out, cost, latency, violations, split by domain. **There is
no side-channel bookkeeping to drift out of sync.** Token numbers are 0 so far
because no real LLM has run — Part 10 fills them in.

In [ ]:
pprint.pp(grx.stats())

## Part 9 — the operator UI: see the system, not the logs

Three surfaces over the *live* session:

- **the trust center** — served at `/` (and `/ui`) by `grx.serve()`: one tab per
  differentiator (ledger, heal economics, the gate's review queue with
  **operable approve/reject**, cited trust with citations that click through to
  the exact ledger rows, routes, trajectories, value ledger) — all recomputed
  from the ledger on every refresh.
- **`grx.save_topology()`** — a self-contained HTML nodes-and-edges map (no CDN,
  no build step): agents, skills, cited tool-trust, route traffic, governance
  hot spots.
- **`grx.serve(port=8077)`** — FastAPI over the live objects, which means
  **Swagger UI at `/docs` for free**, plus `/topology` (re-rendered from the
  ledger on every refresh), `/topology/graph` (JSON for Cytoscape/D3),
  `/stats`, `/trust`, `/gate/pending`, `/gate/why/{id}`, approve/reject —
  operable straight from the Swagger page.

In [ ]:
topo = grx.save_topology()
print("open in a browser:", topo)

In [ ]:
SERVE = False   # flip to True and run this cell, then open http://127.0.0.1:8077/
if SERVE:
    import threading, uvicorn
    threading.Thread(
        target=uvicorn.run,
        kwargs=dict(app=grx.api(), host="127.0.0.1", port=8077, log_level="warning"),
        daemon=True).start()
    print("serving:  http://127.0.0.1:8077/       the trust center (tabs)")
    print("          http://127.0.0.1:8077/docs   /topology   /stats   /trust")

## Part 10 — the full picture: real LLM agents on the substrate

*(Runs only if Ollama is serving `qwen2.5:3b`; skips gracefully otherwise.)*

Two `create_agent` agents — plain LangChain, no graxella imports in their
construction — dropped onto the mesh. Watch for:

1. **hops** — triage hands off to responder via the typed HANDOFF marker;
2. **the drifted tool just works** — `get_shipping_status` heals through the
   rule promoted in Part 4: the agents never see the drift, and `healer_calls`
   stays exactly where it was;
3. **real token accounting** appears in the ledger;
4. **case recall** — later requests are served with earlier ones injected as
   verified experience.

In [ ]:
if OLLAMA:
    from langchain.agents import create_agent
    from langchain_ollama import ChatOllama

    @grx.tool
    def lookup_policy(topic: str) -> str:
        """look up the refund policy for damaged or late items"""
        return ("policy: damaged items refundable within 30 days of delivery; "
                "after 30 days offer store credit only")

    @grx.tool
    def send_email(body: str) -> str:
        """send a friendly apology email response to the customer about their refund"""
        return f"email queued ({len(body)} chars)"

    llm = ChatOllama(model=MODEL, temperature=0)
    triage = create_agent(llm, [check_order, lookup_policy, get_shipping_status],
                          name="triage")
    responder = create_agent(llm, [send_email], name="responder")
    app = grx.mesh([triage, responder])
    print("mesh rebuilt on two real agents — same session, same evidence")
else:
    print("Ollama not detected — skipping Part 10 (everything above already ran).")

In [ ]:
if OLLAMA:
    REQUESTS = [
        "customer says order 1042 espresso machine arrived damaged, wants refund",
        "customer asks about refund for damaged order 2077 desk lamp",
        "customer reports order 1042 damaged again in a second email, refund?",
    ]
    for i, req in enumerate(REQUESTS, 1):
        t0 = time.perf_counter()
        t = app.run_trajectory(req, max_hops=3)
        print(f"[{i}] hops={[h.agent for h in t.hops]}  status={t.status}"
              f"  ({time.perf_counter() - t0:.0f}s)")
        print("    reply:", t.final_response[:120].strip())

In [ ]:
if OLLAMA:
    s = grx.stats()["total"]
    print(f"outcomes={s['count']}  ok_rate={s['ok_rate']}  "
          f"tokens={s['tokens_in']}+{s['tokens_out']}  avg_latency_ms={s['avg_latency_ms']}")
    recalls = app.tracer.events(event_type="recall.injected")
    print(f"case-recall injections: {len(recalls)}")
    print(f"LLM heals during real agent traffic: still {grx.healer_calls} — "
          f"the promoted rule served every drifted call")
    print("\nre-run grx.save_topology() now and the map shows the real traffic.")

## Part 11 — what we measured

| Claim | Measured in this notebook |
|---|---|
| observability for free | every tool call → 2 cited assertions; `grx.stats()` with zero wiring |
| fail once, learn forever | naive fixer: **5** LLM calls for 5 drifted calls; graxella: **1**, then a promoted rule serves everything — including Part 10's real agents |
| governed learning | the heal shipped as a Proposal; a human approved; the verdict + decision are cited ledger assertions; `rulebook.json` is diff-able |
| cited trust | every score carries the assertion ids it was computed from |
| zero-token routing | every route decision: deterministic score + `decision_id`, 0 tokens |
| bounded trajectories | typed status, hop list, budget enforced |
| value ledger | tokens/cost/latency/ok-rate recomputed from assertions alone |
| operator UI | the trust center at `/` (one tab per row of this table), Swagger `/docs`, live `/topology` — all over the running session |

## Beyond this guide

- **`Constitution`** — invariants the gate enforces as hard blocks no human can override (`graxella.Constitution`).
- **PROV-O audit export** — `graxella.audit_export(...)`: W3C provenance JSON-LD of the whole decision graph.
- **Agenda miners** — offline mining of the ledger into new proposals (`graxella.agenda`): rules are *mined from evidence*, not hand-written.
- **`grx.supervisor(agents, model)`** — LLM routing when you want it, same governance, loud fallback.
- **`router="transformer"`** — embedding-based routing, still zero tokens at dispatch.
- **MCP + A2A integration** — `graxella.integrations.mcp`, agent cards from `agent2society`.
- **Control plane** — `graxella.api.control_plane`: data planes buffer locally (WAL) and sync; the control plane being down never affects dispatch.

**Next stop:** [`08_refund_desk.py`](08_refund_desk.py) — the same substrate, condensed into a production-shaped demo with a *real LLM healer*.